#Spark Structured Streaming - Practice
No es posible realizar la práctica por restricciones de escritura en el file system debido al uso de Free Edition

##Step 1 — Create a directory for your streaming data

In [0]:
base_path = "/Workspace/Users/almayo@gmail.com/databricks/stream_demo"

dbutils.fs.mkdirs(base_path)

delta_output = f"{base_path}/delta_output"
checkpoint_dir = f"{base_path}/checkpoint"

dbutils.fs.mkdirs(delta_output)
dbutils.fs.mkdirs(checkpoint_dir)

delta_output, checkpoint_dir

##Step 2 — Create a sample JSON file to simulate incoming data

In [0]:
base_path = "/Volumes/workspace/default/stream_data"

sample_data = """
{"name": "Alice", "age": 30}
{"name": "Bob", "age": 25}
"""
dbutils.fs.put(f"{base_path}/input1.json", sample_data, overwrite=True)


In [0]:
sample_data = """
{"name": "Carlos", "age": 30}
{"name": "Lenka", "age": 25}
"""
dbutils.fs.put(f"{base_path}/input2.json", sample_data, overwrite=True)

##Step 3 — Use readStream to read the folder as a streaming source

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

schema = StructType([
    StructField("name", StringType(), True),
    StructField("age", IntegerType(), True)
])

stream_df = (
    spark.readStream
        .schema(schema)   # mandatory for file streams
        .json(base_path)  # directory to watch
        #.createOrReplaceTempView("workspace.default.sample_data_tmp_view")
)

# This creates a streaming DataFrame, not yet running.

In [0]:
display(stream_df, checkpointLocation = f"{base_path}/checkpoint")

##Step 4 — Start writing to a Delta table inside your base_path directory

###Pre-step: Build subdirectories for output + checkpointing

In [0]:
query = (
    stream_df.writeStream
        .format("delta")
        .trigger(availableNow=True)
        .outputMode("append")
        .option("checkpointLocation", checkpoint_dir)
        .start(delta_output)
)
display(query)

In [0]:
display(query)